In [12]:
# ─── Cell 1: Environment Setup ────────────────────────────────────────────────
import os
import pickle
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Path is now RELATIVE — works on any machine, any OS
# Path().resolve() gives you the folder this notebook is currently running from.
# Then we navigate: scripts/ → data/ → raw/ → resume_data.csv
NOTEBOOK_DIR = Path().resolve()
DATA_PATH    = NOTEBOOK_DIR / "data" / "raw" / "resume_data.csv"
MODELS_DIR   = NOTEBOOK_DIR.parent / "frontend" / "saved_models"

os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Looking for data at : {DATA_PATH}")
print(f"Will save models to : {MODELS_DIR}")
print(f"File exists         : {DATA_PATH.exists()}")

Looking for data at : C:\Users\pipal\OneDrive\Desktop\Intenship\IN Project\career_intelligence_system\scripts\data\raw\resume_data.csv
Will save models to : C:\Users\pipal\OneDrive\Desktop\Intenship\IN Project\career_intelligence_system\frontend\saved_models
File exists         : True


In [13]:
# ─── Cell 2: Load Real Dataset & Extract Features ─────────────────────────────

# Same SKILL_KEYWORDS as frontend/src/nlp_processor.py — keep these in sync
SKILL_KEYWORDS = {
    "python", "java", "c++", "c#", "javascript", "typescript", "html", "css", "scala",
    "react", "angular", "vue", "next.js", "redux", "tailwind css", "figma",
    "node.js", "django", "flask", "fastapi", "graphql", "rest apis",
    "flutter", "dart", "react native", "swift", "kotlin", "sqlite",
    "api integration", "firebase", "state management", "ui/ux design",
    "sql", "mysql", "postgresql", "mongodb", "redis", "oracle",
    "aws", "azure", "gcp", "docker", "kubernetes", "terraform", "jenkins",
    "linux", "bash", "ci/cd", "agile", "scrum",
    "machine learning", "deep learning", "nlp", "computer vision",
    "tensorflow", "pytorch", "pandas", "numpy", "scikit-learn", "llms",
    "etl", "spark", "hadoop", "data modeling", "kafka", "airflow", "snowflake",
    "cybersecurity", "networking", "penetration testing", "firewalls",
    "cryptography", "wireshark", "siem", "ethical hacking",
    "git", "github", "gitlab",
}

# Required skills per domain — matches domain_matcher.py exactly
DOMAIN_SKILLS = {
    "AI & Machine Learning":  ["python", "machine learning", "deep learning", "tensorflow", "pytorch", "sql", "pandas", "numpy"],
    "Mobile App Development": ["flutter", "dart", "react native", "swift", "kotlin", "java", "api integration", "git"],
    "Backend Engineering":    ["python", "java", "node.js", "sql", "rest apis", "git", "linux"],
    "Frontend Engineering":   ["html", "css", "javascript", "react", "vue", "git"],
    "Cloud & DevOps":         ["linux", "bash", "aws", "docker", "kubernetes", "ci/cd", "git"],
    "Data Engineering":       ["sql", "python", "etl", "spark", "hadoop", "data modeling"],
    "Cybersecurity":          ["linux", "networking", "python", "penetration testing", "firewalls", "cryptography"],
}

def extract_features(text, domain=None):
    """
    Extracts structured numerical features from raw resume text.
    This function MUST match the logic in frontend/src/nlp_processor.py
    and frontend/app.py so training and inference use identical features.
    """
    text_lower = text.lower()

    # ── Skills ──────────────────────────────────────────────────────────────
    found = {s for s in SKILL_KEYWORDS if s in text_lower}

    # ── Experience (year-range method — matches Bug 6 fix in app.py) ────────
    years = sorted(set(int(y) for y in re.findall(r"\b(19|20)\d{2}\b", text_lower)))
    exp   = float(years[-1] - years[0]) if len(years) >= 2 else 0.0
    if "senior" in text_lower or "lead" in text_lower:
        exp = max(exp, 5.0)
    exp = min(exp, 10.0)

    # ── Projects (pipe-format + GitHub refs — matches Bug fix in app.py) ───
    proj = max(
        len(re.findall(r'\|\s*[a-z]',     text_lower)),   # "App Name | Flutter, Firebase"
        len(re.findall(r'github\s*[:\.]',  text_lower)),   # "GitHub: username/repo"
        max(len(re.findall(r'\bprojects?\b', text_lower)) - 1, 0),
        0
    )
    proj = min(proj, 15)

    # ── Education ────────────────────────────────────────────────────────────
    if any(w in text_lower for w in ["phd", "master", "m.tech", "mba"]):
        edu = 2
    elif any(w in text_lower for w in ["bachelor", "b.tech", "bsc", "degree"]):
        edu = 1
    else:
        edu = 0

    # ── Certifications ───────────────────────────────────────────────────────
    certs = min(len(re.findall(r"\bcertif\w+", text_lower)), 10)

    # ── Domain fit & gap ─────────────────────────────────────────────────────
    if domain and domain in DOMAIN_SKILLS:
        req      = set(DOMAIN_SKILLS[domain])
        matched  = len(found.intersection(req))
        fit      = (matched / len(req)) * 100
    else:
        # Fallback when no domain label: skill count as proxy
        fit  = min(len(found) * 7, 90.0)
    gap = 100 - fit

    return {
        "experience_years":    round(exp,  1),
        "project_count":       int(proj),
        "education_score":     edu,
        "certification_count": int(certs),
        "domain_fit_score":    round(fit, 1),
        "gap_percentage":      round(gap, 1),
    }

def map_to_domain(job_title):
    """
    Fuzzy matches the highly specific job titles in the dataset 
    to our 7 main training domains.
    """
    title = str(job_title).lower()
    if any(x in title for x in ['machine learning', 'data', 'ai', 'analytics']):
        return "AI & Machine Learning"
    if any(x in title for x in ['ios', 'android', 'mobile', 'flutter']):
        return "Mobile App Development"
    if any(x in title for x in ['frontend', 'web', 'ui', 'react', 'design']):
        return "Frontend Engineering"
    if any(x in title for x in ['cloud', 'devops', 'aws', 'azure']):
        return "Cloud & DevOps"
    if any(x in title for x in ['security', 'cyber', 'network']):
        return "Cybersecurity"
    
    # Fallback to Backend for general software or unmapped roles
    return "Backend Engineering"


# ── Load real data ────────────────────────────────────────────────────────────
try:
    raw_df = pd.read_csv(DATA_PATH)
    
    # Clean column names (Removes hidden \ufeff characters from the start of column names)
    raw_df.columns = raw_df.columns.str.replace('\ufeff', '')
    
    print(f"Loaded {len(raw_df)} resumes from dataset.")
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nData file not found at: {DATA_PATH}\n"
    )

# ── Extract features from every resume ───────────────────────────────────────
print("Extracting features from resume text...")
records = []

# List of columns to combine to simulate a full resume text
text_columns = [
    'career_objective', 'skills', 'degree_names', 
    'major_field_of_studies', 'positions', 'responsibilities'
]

for _, row in raw_df.iterrows():
    # 1. Safely combine all available text fields into one giant paragraph
    combined_text = []
    for col in text_columns:
        val = row.get(col, "")
        if pd.notna(val):  # Ensure we don't append NaN values
            combined_text.append(str(val))
    text = " ".join(combined_text)
    
    # 2. Get the category (Job Position Name) and map it to a Domain
    cat = str(row.get("job_position_name", ""))
    domain = map_to_domain(cat)
    
    # 3. Extract Features
    feat = extract_features(text, domain)
    feat["category"] = cat
    records.append(feat)

df = pd.DataFrame(records)
print(f"Feature extraction complete. Shape: {df.shape}")
print(df.head().to_string())

Loaded 9544 resumes from dataset.
Extracting features from resume text...
Feature extraction complete. Shape: (9544, 7)
   experience_years  project_count  education_score  certification_count  domain_fit_score  gap_percentage                                                        category
0               0.0              0                1                    0              28.6            71.4                                        Senior Software Engineer
1               5.0              0                0                    0              12.5            87.5                                  Machine Learning (ML) Engineer
2               0.0              0                1                    0              28.6            71.4  Executive/ Senior Executive- Trade Marketing, Hygiene Products
3               0.0              0                0                    1               0.0           100.0                                  Business Development Executive
4               5.0      

In [14]:
# ─── Cell 3: Create Realistic Training Labels & Prep Data ─────────────────────

def make_label(row):
    score = (
        (row["domain_fit_score"] / 100.0)        * 35 +
        min(row["experience_years"] / 5.0, 1.0)  * 25 +
        (row["education_score"]   / 2.0)          * 15 +
        min(row["project_count"]  / 3.0, 1.0)    * 15 +
        min(row["certification_count"] / 2.0,1.0) * 10
    )
    # Add small Gaussian noise so the model doesn't learn a perfect boundary
    noise = np.random.normal(0, 4)
    # Threshold for shortlisting (adjust this if your rate is too high/low)
    return 1 if (score + noise) >= 42 else 0 

np.random.seed(42)   # reproducibility
df["shortlisted"] = df.apply(make_label, axis=1)

rate = df["shortlisted"].mean() * 100
print(f"Shortlisting rate: {rate:.1f}%  (healthy range: 30–50%)")

# ── Prepare Features (X) and Target (y) ──
# Machine learning models only understand numbers. We must drop the text 'category'
X = df.drop(columns=["category", "shortlisted"])
y = df["shortlisted"]

# ── Train/Test Split ──
# Split data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Feature Scaling ──
# Standardize the numbers so 'experience' (0-10) and 'domain_fit' (0-100) carry equal weight
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\n✅ Data scaled and split successfully.")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape:     {X_test_scaled.shape}")

Shortlisting rate: 10.4%  (healthy range: 30–50%)

✅ Data scaled and split successfully.
Training set shape: (7635, 6)
Test set shape:     (1909, 6)


In [15]:
# ─── Cell 4: Train Model & Evaluate ────────────────────────────────────────

# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    random_state=42, 
    n_jobs=-1
)

print("Training Random Forest model...")
rf_model.fit(X_train_scaled, y_train)

# Evaluate the model
y_pred = rf_model.predict(X_test_scaled)
print("\n✅ Model Training Complete!")
print("Model Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Training Random Forest model...

✅ Model Training Complete!
Model Accuracy: 0.968

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98      1710
           1       0.92      0.76      0.83       199

    accuracy                           0.97      1909
   macro avg       0.94      0.88      0.91      1909
weighted avg       0.97      0.97      0.97      1909



In [16]:
# ─── Cell 5: Save Models for Frontend ────────────────────────────────────────

# The frontend app.py attempts to load a vectorizer.pkl. 
# While our Random Forest uses extracted numerical features, we create a basic 
# TF-IDF vectorizer here for text extraction purposes in your NLP Processor 
# and to ensure the frontend doesn't crash on load.

dummy_corpus = [
    "python java html css javascript react frontend backend",
    "machine learning data science artificial intelligence sql python",
    "aws azure docker kubernetes devops ci cd cloud",
    "cybersecurity network firewall security penetration testing",
    "mobile flutter ios android swift kotlin react native"
]
vectorizer = TfidfVectorizer(max_features=500)
vectorizer.fit(dummy_corpus)

# --- Save the Models ---
# First, ensure the target directory actually exists
import os
os.makedirs("../frontend/saved_models", exist_ok=True)

rf_path = "../frontend/saved_models/rf_classifier.pkl"
scaler_path = "../frontend/saved_models/scaler.pkl"
vec_path = "../frontend/saved_models/vectorizer.pkl"

with open(rf_path, "wb") as f:
    pickle.dump(rf_model, f)

with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
    
with open(vec_path, "wb") as f:
    pickle.dump(vectorizer, f)

print("✅ Success! Models saved to:")
print(f" - {rf_path}")
print(f" - {scaler_path}")
print(f" - {vec_path}")
print("\n🚀 The training pipeline is complete. You can now start your frontend application!")

✅ Success! Models saved to:
 - ../frontend/saved_models/rf_classifier.pkl
 - ../frontend/saved_models/scaler.pkl
 - ../frontend/saved_models/vectorizer.pkl

🚀 The training pipeline is complete. You can now start your frontend application!
